# SAM3 GPU Server Inference

This notebook runs Segment Anything 3 on a CUDA-enabled GPU server.

## Environment Setup

### Configure HuggingFace Token

To pull Segment Anything 3 weights, you need a HuggingFace Access Token with approved access to the SAM 3 checkpoints.

- Request access to the SAM 3 checkpoints on the official Hugging Face [repo](https://huggingface.co/facebook/sam2-hiera-large)
- Create a `.env` file in the project root with your token:
  ```
  HF_TOKEN=hf_your_token_here
  ```

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
project_root = Path.cwd().parent.parent
env_path = project_root / ".env"
load_dotenv(env_path)

hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN not found. Please create a .env file with your HuggingFace token.")

os.environ["HF_TOKEN"] = hf_token
print("✓ HuggingFace token loaded successfully")

### Check GPU Availability

Verify that CUDA GPU is available and properly configured.

In [ ]:
!nvidia-smi

In [ ]:
import torch
import torchvision

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA is available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU compute capability: {torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}")
else:
    raise RuntimeError("CUDA GPU not available. This notebook requires a CUDA-enabled GPU.")

### Install SAM3 and Dependencies

Clone the official SAM3 repository and install it with required dependencies.

In [ ]:
# Check if sam3 is already cloned
sam3_dir = Path.cwd() / "sam3"

if not sam3_dir.exists():
    print("Cloning SAM3 repository...")
    !git clone https://github.com/facebookresearch/sam3.git
    print("✓ SAM3 repository cloned")
else:
    print("✓ SAM3 repository already exists")

# Install SAM3 and dependencies
%cd sam3
%pip install -e ".[notebooks]"
%cd ..

In [ ]:
# Install additional dependencies for visualization and interaction
%pip install -q supervision jupyter_bbox_widget python-dotenv

### Download Example Data

Download sample images for testing. You can replace these with your own images.

In [ ]:
# Create data directory
data_dir = Path.cwd() / "data"
data_dir.mkdir(exist_ok=True)

# Download example images
images = [
    "solvay_conference_1927.jpg",
    "birds.jpg",
    "pills.jpg",
    "eggs.jpg",
    "traffic_jam.jpg",
    "airport.jpg",
    "basketball_game.jpg",
    "dog-2.jpeg"
]

for img in images:
    img_path = data_dir / img
    if not img_path.exists():
        !wget -q https://media.roboflow.com/notebooks/examples/{img} -P {data_dir}
        print(f"✓ Downloaded {img}")
    else:
        print(f"✓ {img} already exists")

---
**⚠️ Important: Restart the kernel before running cells below if you just installed SAM3.**

---

## Load SAM3 Image Predictor

On Ampere GPUs (compute capability ≥ 8), we enable TensorFloat-32 (TF32) for matrix multiplications and convolutions. This allows PyTorch to use tensor cores to accelerate FP32 computations while maintaining similar numerical accuracy.

In [ ]:
import torch
from pathlib import Path

# Enable autocast for better performance
torch.autocast(device_type="cuda", dtype=torch.bfloat16).__enter__()

# Enable TF32 on Ampere GPUs for better performance
if torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("✓ TF32 enabled for Ampere GPU")
else:
    print("ℹ GPU compute capability < 8, TF32 not available")

In [ ]:
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

# Build the model
bpe_path = Path.cwd() / "sam3" / "sam3" / "assets" / "bpe_simple_vocab_16e6.txt.gz"
model = build_sam3_image_model(bpe_path=str(bpe_path))
processor = Sam3Processor(model, confidence_threshold=0.3)

print("✓ SAM3 model loaded successfully")

## Utility Functions for Visualization

In [ ]:
import supervision as sv
import numpy as np

def from_sam(sam_result: dict) -> sv.Detections:
    """Convert SAM3 results to Supervision Detections format."""
    xyxy = sam_result["boxes"].to(torch.float32).cpu().numpy()
    confidence = sam_result["scores"].to(torch.float32).cpu().numpy()

    mask = sam_result["masks"].to(torch.bool)
    mask = mask.reshape(mask.shape[0], mask.shape[2], mask.shape[3]).cpu().numpy()

    return sv.Detections(
        xyxy=xyxy,
        confidence=confidence,
        mask=mask
    )

In [ ]:
import supervision as sv
from PIL import Image
from typing import Optional

# Define color palette for visualizations
COLOR = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
    "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])


def annotate(image: Image.Image, detections: sv.Detections, label: Optional[str] = None) -> Image.Image:
    """Annotate image with detection results."""
    text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)

    mask_annotator = sv.MaskAnnotator(
        color=COLOR,
        color_lookup=sv.ColorLookup.INDEX,
        opacity=0.6
    )
    box_annotator = sv.BoxAnnotator(
        color=COLOR,
        color_lookup=sv.ColorLookup.INDEX,
        thickness=1
    )
    label_annotator = sv.LabelAnnotator(
        color=COLOR,
        color_lookup=sv.ColorLookup.INDEX,
        text_scale=0.4,
        text_padding=5,
        text_color=sv.Color.BLACK,
        text_thickness=1
    )

    annotated_image = image.copy()
    annotated_image = mask_annotator.annotate(annotated_image, detections)
    annotated_image = box_annotator.annotate(annotated_image, detections)

    if label:
        labels = [
            f"{label} {confidence:.2f}"
            for confidence in detections.confidence
        ]
        annotated_image = label_annotator.annotate(annotated_image, detections, labels)

    return annotated_image

## SAM3 Text Prompt

Segment objects using text descriptions.

In [ ]:
from PIL import Image
from IPython.display import display

PROMPT = "taxi"
IMAGE_PATH = data_dir / "traffic_jam.jpg"

image = Image.open(IMAGE_PATH).convert("RGB")
inference_state = processor.set_image(image)
inference_state = processor.set_text_prompt(state=inference_state, prompt=PROMPT)

detections = from_sam(sam_result=inference_state)
detections = detections[detections.confidence > 0.5]

print(f"There are {len(detections)} {PROMPT} objects detected in the image.\n")
annotated = annotate(image, detections, label=PROMPT)
display(annotated)

## SAM3 Box Prompt

Segment objects by drawing bounding boxes.

In [ ]:
import base64
from io import BytesIO
from PIL import Image

OBJECTS = ['positive', 'negative']

def encode_image_pillow(image: Image.Image) -> str:
    """Encode PIL Image to base64 string for widget display."""
    buffer = BytesIO()
    image.save(buffer, format="JPEG")
    image_bytes = buffer.getvalue()
    encoded = base64.b64encode(image_bytes).decode("utf-8")
    return "data:image/jpeg;base64," + encoded

When prompting SAM 3 with bounding boxes, the model expects boxes in the `xcycwh` format (`x_center`, `y_center`, `width`, `height`), and the coordinates must be normalized. The code below handles the conversion to this format.

In [ ]:
import numpy as np

def get_normalized_boxes(bboxes, label, resolution_wh):
    """Convert bounding boxes to normalized xcycwh format."""
    width, height = resolution_wh
    boxes = [
        [b["x"] + b["width"] / 2, b["y"] + b["height"] / 2, b["width"], b["height"]]
        for b in bboxes
        if b["label"] == label
    ]
    if not boxes:
        return np.empty((0, 4), dtype=np.float32)
    scale = np.array([width, height, width, height], dtype=np.float32).reshape(1,-1)
    return np.array(boxes, dtype=np.float32) / scale

In [ ]:
from PIL import Image
from jupyter_bbox_widget import BBoxWidget

image_path = data_dir / "birds.jpg"
image = Image.open(image_path).convert("RGB")

widget = BBoxWidget(classes=OBJECTS)
widget.image = encode_image_pillow(image)
widget

In [ ]:
# After drawing boxes in the widget above, run this to see results
# Uncomment and run when ready:

# positive_boxes = get_normalized_boxes(widget.bboxes, "positive", image.size)
# negative_boxes = get_normalized_boxes(widget.bboxes, "negative", image.size)

# inference_state = processor.set_image(image)
# inference_state = processor.set_box_prompt(
#     state=inference_state,
#     positive_boxes=positive_boxes,
#     negative_boxes=negative_boxes
# )

# detections = from_sam(sam_result=inference_state)
# detections = detections[detections.confidence > 0.3]

# print(f"Detected {len(detections)} objects")
# annotated = annotate(image, detections)
# display(annotated)

## Load SAM3 Interactive Image Predictor

Segment Anything Model 3 (SAM 3) predicts object masks given prompts that indicate the desired object. The model first converts the image into an image embedding that allows high quality masks to be efficiently produced from a prompt.

`SAM3InteractiveImagePredictor.predict` takes the following arguments:

- `point_coords` - `[np.ndarray or None]` - a `Nx2` array of point prompts to the model. Each point is in `(X,Y)` in pixels.
- `point_labels` - `[np.ndarray or None]` - a length `N` array of labels for the point prompts. `1` indicates a foreground point and `0` indicates a background point.
- `box` - `[np.ndarray or None]` - a length `4` array given a box prompt to the model, in `[x_min, y_min, x_max, y_max]` format.
- `mask_input` - `[np.ndarray]` - a low resolution mask input to the model, typically coming from a previous prediction iteration. Has form `1xHxW`, where for SAM, `H=W=256`.
- `multimask_output` - `[bool]` - if true, the model will return three masks. For ambiguous input prompts (such as a single click), this will often produce better masks than a single prediction. If only a single mask is needed, the model's predicted quality score can be used to select the best mask. For non-ambiguous prompts, such as multiple input prompts, `multimask_output=False` can give better results.
- `return_logits` - `[bool]` - if true, returns un-thresholded masks logits instead of a binary mask.
- `normalize_coords` - `[bool]` - if true, the point coordinates will be normalized to the range `[0,1]` and point_coords is expected to be wrt. image dimensions.

In [ ]:
# Load model with interactive mode enabled
model = build_sam3_image_model(enable_inst_interactivity=True)
processor = Sam3Processor(model, confidence_threshold=0.3)

print("✓ SAM3 interactive model loaded successfully")

In [ ]:
def from_sam_interactive(sam_result: tuple[np.ndarray, np.ndarray]) -> sv.Detections:
    """Convert interactive SAM3 results to Supervision Detections format."""
    masks, scores = sam_result

    if masks.shape[0] != 1:
        masks = np.squeeze(masks)

    return sv.Detections(
        xyxy=sv.mask_to_xyxy(masks=masks),
        mask=masks.astype(bool),
        confidence=np.squeeze(scores)
    )

In [ ]:
import numpy as np

def get_xy_points(boxes, label):
    """Extract center points from bounding boxes."""
    points = [
        [b["x"] + b["width"] / 2, b["y"] + b["height"] / 2]
        for b in boxes
        if b["label"] == label
    ]
    if not points:
        return np.empty((0, 2), dtype=np.float32)
    return np.array(points, dtype=np.float32)


def get_xyxy_boxes(boxes, label):
    """Convert boxes to xyxy format."""
    boxes = [
        [b["x"], b["y"], b["x"] + b["width"], b["y"] + b["height"]]
        for b in boxes
        if b["label"] == label
    ]
    if not boxes:
        return np.empty((0, 4), dtype=np.float32)
    return np.array(boxes, dtype=np.float32)

## Interactive SAM3 Box Prompt

In [ ]:
from PIL import Image
from jupyter_bbox_widget import BBoxWidget

image_path = data_dir / "dog-2.jpeg"
image = Image.open(image_path).convert("RGB")

widget = BBoxWidget(classes=OBJECTS)
widget.image = encode_image_pillow(image)
widget

In [ ]:
# After drawing boxes in the widget above, run this to see results
# Uncomment and run when ready:

# positive_boxes = get_xyxy_boxes(widget.bboxes, "positive")
# negative_boxes = get_xyxy_boxes(widget.bboxes, "negative")

# inference_state = processor.set_image(image)

# # Process positive boxes
# for box in positive_boxes:
#     masks, scores = processor.predict(
#         inference_state=inference_state,
#         box=box,
#         multimask_output=False
#     )
#     detections = from_sam_interactive((masks, scores))
#     annotated = annotate(image, detections)
#     display(annotated)

## Interactive SAM3 Point Prompt

In [ ]:
from PIL import Image
from jupyter_bbox_widget import BBoxWidget

image_path = data_dir / "dog-2.jpeg"
image = Image.open(image_path).convert("RGB")

widget = BBoxWidget(classes=OBJECTS)
widget.image = encode_image_pillow(image)
widget

In [ ]:
# After clicking points in the widget above, run this to see results
# Uncomment and run when ready:

# positive_points = get_xy_points(widget.bboxes, "positive")
# negative_points = get_xy_points(widget.bboxes, "negative")

# if len(positive_points) > 0 or len(negative_points) > 0:
#     # Combine points and labels
#     all_points = np.vstack([positive_points, negative_points]) if len(negative_points) > 0 else positive_points
#     point_labels = np.array(
#         [1] * len(positive_points) + [0] * len(negative_points),
#         dtype=np.int32
#     )
    
#     inference_state = processor.set_image(image)
#     masks, scores = processor.predict(
#         inference_state=inference_state,
#         point_coords=all_points,
#         point_labels=point_labels,
#         multimask_output=True
#     )
    
#     detections = from_sam_interactive((masks, scores))
#     annotated = annotate(image, detections)
#     display(annotated)
# else:
#     print("Please add some points using the widget above")

---

## Summary

This notebook demonstrates how to run SAM3 on a CUDA-enabled GPU server with:

- **Text prompts**: Segment objects by describing them in natural language
- **Box prompts**: Segment objects by drawing bounding boxes
- **Point prompts**: Segment objects by clicking points (foreground/background)
- **Interactive mode**: Real-time segmentation with user feedback

The notebook is optimized for CUDA GPUs and includes TF32 acceleration for Ampere and newer architectures.

---